# **Learned features в CNN: Практика**

Добро пожаловать в практическую часть по первой части модуля [«Анализ CNN-based моделей»](https://ai-interpretability.school/learn/9/0/0)!

Повторим пройденные тезисы:

1. Сверточные нейронные сети извлекают паттерны изображения благодаря skip-connections.
2. Извлеченные из изображения структуры для обученной сверточной сети можно увидеть, пропустив пример через слои свертки с весами последовательно.
3. Сверточные нейронные сети способны извлекать понятные человеку концепции из данных.

Чтобы убедиться в каждом, перейдем к практике.

**Задание 1. Какой из тезисов 1–3, предложенных в начале, содержит ошибку?**


In [ ]:
!pip install torch torchvision -q


In [ ]:
import torch
import requests
import numpy as np
from tqdm import tqdm
from io import BytesIO
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

Будем работать с ResNet18, также известной как Residual Network — достаточно популярной архитектурой для задачи классификации. Основная особенность ResNet — использование остаточных связей (skip connections), что позволяет эффективно обучать очень глубокие сети, избегая проблемы затухания градиентов.

В ResNet18, согласно названию, используется 18 слоев. Размер входного слоя сети 224 x 224. После, грубо говоря, принятия изображения в сеть последовательно применяются слои свертки с MaxPooling, Batch Normalization и функциями активации.




## Архитектура ResNet18

Картинка 1.
![ResNet-18](https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/resnet18_architecture.png)

Источник: [Dive into Deep Learning](https://d2l.ai/chapter_convolutional-modern/resnet.html) (Zhang, Lipton, Li и Smola), [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Resnet-18_architecture_(rotated).svg), CC BY-SA 4.0.

![ResNet-18, блоки](https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/resnet18_basic_blocks.png)
Картинка 2.

Загрузим предобученную модель из PyTorch и посмотрим на ее архитектуру.

In [ ]:
# Загрузка модели
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

#Рассмотрение архитектуры
model

**Задание 2. Какая функция активации применяется в ResNet18?**  
**Задание 3. Сколько слоев свертки содержит данная архитектура ResNet18?**

Вспомним еще несколько деталей из модуля:

- Первые (от входного слоя) сверточные слои изучают очень маленькие составляющие: края и цветовые переходы.
- Следующие слои изучают отдельные текстуры.
- Ближе к последним сверточным слоям в извлекаемой информации просматриваются паттерны.
- В конце можно увидеть изображения.

Также мы зафиксировали, что эти концепции можно увидеть в любой обученной сверточной нейронной сети, применяя слои свертки с их весами к данным последовательно.

Тогда сформулируем, что нам нужно:

1. Рассмотреть обученную сеть $net(img)$, содержащую $n$ слоев свертки с фиксированными весами $w_i, i=\vec{1, n}$.
2. Для каждого слоя внутри сети:
    - если он является сверточным, извлечь слой с его параметрами;
    - иначе пойти дальше.

Для решения задачи мы могли бы написать простой цикл, но архитектура ResNet18 предполагает, что свертки являются частями последовательного (*Sequential*) слоя. Учитывая это, наш алгоритм придется немного преобразовать:

1. Рассмотреть обученную сеть $net(img)$, содержащую $n$ слоев свертки с фиксированными весами $w_i, i=\vec{1, n}$.
2. Для каждого слоя внутри сети:
    - если он является сверточным, извлечь слой с его параметрами;
    - иначе, если слой является Sequential:
        - рассмотреть каждую его составляющую;
        - если составляющая является сверточным слоем, извлечь ее с ее параметрами;
        - иначе пойти дальше.

И таким способом нам надо пройтись по сети. Сделаем ниже в коде.

В действительности, для визуализации мы извлечем не все сверточные слои: свертки из подблоков *downsample* мы пропустим.

![temp-Imagermy-OBM.avif](https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/assets/temp-Imagermy-OBM.avif)

 Оператор `print` этого не отражает, но свертки в downsample применяются не линейно, а параллельно сверткам выше в базовом блоке (`BasicBlock`). Их выход складывается с выходом основной ветви блока и уходит в следующий блок, поэтому, не извлекая их отдельно, мы не теряем их вклад в следующие карты.

In [ ]:
# Списки для извлечения сверточных слоев и их весов
conv_weights = [] # список для весов слоя
conv_layers = [] # список для хранения самих слоев

counter = 0 # счетчик для того, чтобы убедиться, что мы извлекли все слои

# сохраним все компоненты модели в список
model_children = list(model.children())


for i in range(len(model_children)):
    if type(model_children[i]) == nn.Conv2d:
        counter+=1
        conv_weights.append(model_children[i].weight)
        conv_layers.append(model_children[i])
    elif type(model_children[i]) == nn.Sequential:
      for j in range(0, len(model_children[i])):
        for child in model_children[i][j].children():
          if type(child) == nn.Conv2d:
                    counter+=1
                    conv_weights.append(child.weight)
                    conv_layers.append(child)
          elif type(child) == nn.Sequential and len(list(child.children())) != 0:
            for k in range(0, len(list(child.children()))):
              if type(list(child.children())[k]) == nn.Conv2d:
                counter+=1
                conv_weights.append(list(child.children())[k].weight)
                conv_layers.append(list(child.children())[k])


print(f'Извлечено слоев: {counter}')

**Задание 4.** Исправьте код ниже так, чтобы он извлек все сверточные слои, кроме тех, что содержатся в downsample подблоках. Сколько слоев у вас получилось после исправления?

In [ ]:
# Задание 4: цикл ниже берет все свертки сети, как и ячейка выше, — 20 слоев.
# Исправьте условие так, чтобы в список не попали свертки подблоков downsample
# (в имени такого модуля есть слово 'downsample').
conv_layers = []  # список для хранения самих слоев
for name, module in model.named_modules():
    if isinstance(module, nn.Conv2d):  # <- исправьте условие
        conv_layers.append(module)
conv_weights = [layer.weight for layer in conv_layers]  # список для весов слоев
counter = len(conv_layers)

print(f'Извлечено слоев: {counter}')

**Входное изображение.**

Загрузим конкретный пример $x_0$.

Автор фото хрюшки: [Brett Sayles, Pexels](https://www.pexels.com/photo/4966539/), лицензия Pexels.

In [ ]:
# Загрузка изображения
url = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/piglet.jpg'


image_bytes = requests.get(url).content
image = Image.open(BytesIO(image_bytes))
plt.imshow(image)
plt.show()

In [ ]:
print(f"Исходный размер изображения: {image.size}")

In [ ]:
np.array(image).shape

Pillow открывает снимок в том режиме, в каком он сохранен: у jpg это обычно три канала RGB, у png часто четыре, RGBA, где четвертый отвечает за прозрачность пикселя. Сеть ждет ровно три канала, поэтому приведите изображение к RGB явно: тогда код сработает со снимком любого формата.

In [ ]:
#Ваш код здесь

На вход ResNet18 может быть подано изображение любого разрешения. Проходя свертки последовательно и благодаря `AdaptiveAvgPool2d` изображение обработается в любом случае. Однако, чтобы карты признаков вышли крупнее и их было удобно рассматривать, мы растянем снимок до 1224x1224 и нормализуем.

Обратите внимание, что mean и std мы задаем сразу, не вычисляя. Для моделей, обученных на ImageNet, эти значения — обычная практика. Они были рассчитаны на основе изображений датасета.

In [ ]:
#Средние
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((1224, 1224)),  # Изменяем размер изображения
    transforms.ToTensor(),  # Конвертируем картинку в PyTorch тензор
    transforms.Normalize(mean=mean, std=std)  # Нормализуем картинку
])

image = transform(image)
print(f"Размер изображения после изменения размера: {image.shape}")

In [ ]:
# Переместим модель на GPU если это возможно
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = model.to(device)
image = image.to(device)

# Определим списки для извлечения feature maps
feature_maps = []  # Список для feature maps
layer_names = [str(number+1)+'_'+str(layer) for number, layer in enumerate(conv_layers)]  # Список для layer names

# На каждую свертку из conv_layers повесим хук: во время обычного прямого прохода сети
# он сохранит выход свертки. Так карты получаются из настоящего вычисления ResNet18 —
# вместе с BatchNorm, ReLU, пулингом и остаточными связями между свертками, —
# а не из цепочки одних сверток, приложенных друг к другу.
hooks = [layer.register_forward_hook(lambda module, inp, out: feature_maps.append(out.detach()))
         for layer in conv_layers]
model.eval()
with torch.no_grad():
    model(image.unsqueeze(0))  # прямой проход по батчу из одной картинки
for hook in hooks:
    hook.remove()
print(f'Карт признаков: {len(feature_maps)}')



In [ ]:
# Предобработка feature maps к визуализируемому виду
processed_feature_maps = [] # Список для хранения полученных карт активации
for feature_map in feature_maps:
	feature_map = feature_map.squeeze(0) # Удаляем размерность батча
	mean_feature_map = torch.sum(feature_map, 0) / feature_map.shape[0] # Вычисляем среднее по каналам
	processed_feature_maps.append(mean_feature_map.data.cpu().numpy())

Наконец, отобразим получившиеся результаты.

In [ ]:
# Отобразим карты признаков
fig = plt.figure(figsize=(30, 30))
for i in range(len(processed_feature_maps)):
	ax = fig.add_subplot(5, 4, i + 1)
	ax.imshow(processed_feature_maps[i])
	ax.axis("off")
	ax.set_title(layer_names[i].split('(')[0], fontsize=25)


**Задание 5. Почему хрюша получилась зеленой?**

**Задание 6. Поэкспериментируйте с `transforms.Resize` в `transform`. Что меняется в результате?**